# Tracking a timed trajectory: notebook

The room robot of RO-001 (wheel separation 0.2 m, each wheel at most 0.6 m/s), driven by its forward speed v and turn rate ω.
Its timetable: from (0, 0) facing along x at 0.3 m/s, 1 m straight, a 90° left arc of radius 1 m, 1 m straight.
The robot starts 0.2 m behind and 0.1 m to the left. The tracker runs at 20 Hz; the robot is stepped every 0.01 s.
Every number in the Note is computed here; the model and the trackers are in `images/trackkit.py`, which every figure script also uses.

In [1]:
import sys
sys.path.insert(0, "images")
import numpy as np
from trackkit import *
np.set_printoptions(precision=4, suppress=True)
print(f"end of the first straight {T1:.3f} s, end of the arc {T2:.3f} s, end of the timetable {T_END:.3f} s")
print("timetable at 0, 3.333, 6, 8.57, 11.9 s:")
for t in (0, T1, 6, T2, T_END):
    x, y, th, v, w = ref(t)
    print(f"  t = {t:6.3f} s: ({x:.3f}, {y:.3f}), facing {np.degrees(th):5.1f} deg, v = {v} m/s, w = {w:.1f} rad/s")

end of the first straight 3.333 s, end of the arc 8.569 s, end of the timetable 11.903 s
timetable at 0, 3.333, 6, 8.57, 11.9 s:
  t =  0.000 s: (0.000, 0.000), facing   0.0 deg, v = 0.3 m/s, w = 0.0 rad/s
  t =  3.333 s: (1.000, 0.000), facing   0.0 deg, v = 0.3 m/s, w = 0.0 rad/s
  t =  6.000 s: (1.717, 0.303), facing  45.8 deg, v = 0.3 m/s, w = 0.3 rad/s
  t =  8.569 s: (2.000, 1.000), facing  90.0 deg, v = 0.3 m/s, w = 0.3 rad/s
  t = 11.903 s: (2.000, 2.000), facing  90.0 deg, v = 0.3 m/s, w = 0.0 rad/s


## 1. The opening problem (Note, Section 1)

In [2]:
runs = {"replay": simulate(replay), "path follower": simulate(path_follower), "Kanayama": simulate(kanayama)}
for name, r in runs.items():
    xe, ye, the, d = errors(r)
    within = r["t"][np.argmax(d < 0.01)] if (d < 0.01).any() else None
    print(f"{name:14s} start {d[0]:.4f} m, end {d[-1]:.4f} m from the timetable; x_e at the end {xe[-1]:.4f} m;"
          f" first within 1 cm at {within} s; a wheel limit hit: {r['clipped']}")
late = simulate(path_follower, T=T_END + 2)
t_arr = late["t"][np.argmax(late["y"] >= 2.0)]
print(f"path follower reaches the end (2, 2) at {t_arr:.2f} s: {t_arr - T_END:.2f} s late")

replay         start 0.2236 m, end 0.2187 m from the timetable; x_e at the end -0.0943 m; first within 1 cm at None s; a wheel limit hit: False
path follower  start 0.2236 m, end 0.2050 m from the timetable; x_e at the end 0.2050 m; first within 1 cm at None s; a wheel limit hit: False
Kanayama       start 0.2236 m, end 0.0001 m from the timetable; x_e at the end 0.0000 m; first within 1 cm at 3.35 s; a wheel limit hit: False
path follower reaches the end (2, 2) at 12.59 s: 0.69 s late


## 2. Replaying the timetable keeps the start error (Note, Section 2)
In continuous time, two robots with the same heading and the same (v, ω) keep a constant offset. The 20 Hz command hold changes it slightly.

In [3]:
xe, ye, the, d = errors(runs["replay"])
print("replay: distance at 0, 3.33, 8.57, 11.9 s:", [round(d[int(round(t / DT))], 4) for t in (0, 3.33, 8.57, T_END)])
print("heading difference at the end (deg):", round(np.degrees(the[-1]), 2))
print("replay: smallest and largest distance over the run:", round(d.min(), 4), round(d.max(), 4))

replay: distance at 0, 3.33, 8.57, 11.9 s: [np.float64(0.2236), np.float64(0.2236), np.float64(0.2149), np.float64(0.2187)]
heading difference at the end (deg): -0.24
replay: smallest and largest distance over the run: 0.2149 0.2236


## 3. The error in the robot's own frame (Note, Section 3)

In [4]:
print("facing 0 deg :", np.round(error_frame((0, 0, 0), (0.2, -0.1, 0)), 4))
print("facing 45 deg:", np.round(error_frame((0, 0, np.pi / 4), (0.2, -0.1, np.pi / 4)), 4))
c = np.cos(np.pi / 4)
print("by hand: x_e = cos45 * 0.2 + sin45 * (-0.1) =", round(c * 0.2 + c * -0.1, 4), "; y_e = -sin45 * 0.2 + cos45 * (-0.1) =", round(-c * 0.2 + c * -0.1, 4))

facing 0 deg : [ 0.2 -0.1  0. ]
facing 45 deg: [ 0.0707 -0.2121  0.    ]
by hand: x_e = cos45 * 0.2 + sin45 * (-0.1) = 0.0707 ; y_e = -sin45 * 0.2 + cos45 * (-0.1) = -0.2121


The error's rates at the start, with Kanayama's first command (v = 0.5 m/s, ω = −0.48 rad/s); a numeric check by a small step.

In [5]:
pose0, xe0, ye0, th0 = START, 0.2, -0.1, 0.0
v0, w0 = kanayama(0.0, pose0)
print("first command v, w:", round(v0, 4), round(w0, 4), "; wheels right, left:", round(v0 + w0 * 0.1, 3), round(v0 - w0 * 0.1, 3))
xd = w0 * ye0 - v0 + V_R * np.cos(th0); yd = -w0 * xe0 + V_R * np.sin(th0); thd = 0 - w0
print("rates from the formulas: x_e'", round(xd, 4), " y_e'", round(yd, 4), " th_e'", round(thd, 4))
h = 1e-6   # one tiny step of both robots with these commands
p1 = (pose0[0] + v0 * h, pose0[1], pose0[2] + w0 * h)
e1 = error_frame(p1, (V_R * h, 0, 0))
print("rates by a tiny step:      ", np.round((np.array(e1) - [xe0, ye0, th0]) / h, 4))

first command v, w: 0.5 -0.48 ; wheels right, left: 0.452 0.548
rates from the formulas: x_e' -0.152  y_e' 0.096  th_e' 0.48
rates by a tiny step:       [-0.152  0.096  0.48 ]


## 4. Kanayama's tracker (Note, Section 4)

In [6]:
r = runs["Kanayama"]
xe, ye, the, d = errors(r)
for t in (0, 0.5, 1, 2, 3, 3.35, 6, T_END):
    i = int(round(t / DT))
    print(f"t = {t:5.2f} s: x_e {100 * xe[i]:6.2f} cm, y_e {100 * ye[i]:6.2f} cm, th_e {np.degrees(the[i]):5.2f} deg,"
          f" distance {100 * d[i]:5.2f} cm, v {r['v'][i]:.3f}, w {r['w'][i]:.3f}")
wheels = np.maximum(np.abs(r["v"] + r["w"] * 0.1), np.abs(r["v"] - r["w"] * 0.1))
print("largest wheel speed:", round(wheels.max(), 3), "m/s; largest heading error:", round(np.degrees(np.abs(the).max()), 2), "deg")
i335 = int(round(3.35 / DT))
print("largest distance from 3.35 s on:", round(d[i335:].max(), 4), "m; time of the largest heading error:", r["t"][np.abs(the).argmax()], "s")

t =  0.00 s: x_e  20.00 cm, y_e -10.00 cm, th_e  0.00 deg, distance 22.36 cm, v 0.500, w -0.480
t =  0.50 s: x_e  12.71 cm, y_e  -6.87 cm, th_e  6.65 deg, distance 14.44 cm, v 0.425, w -0.052
t =  1.00 s: x_e   7.61 cm, y_e  -5.05 cm, th_e  6.68 deg, distance  9.13 cm, v 0.374, w 0.037
t =  2.00 s: x_e   2.62 cm, y_e  -2.47 cm, th_e  3.93 deg, distance  3.60 cm, v 0.326, w 0.046
t =  3.00 s: x_e   0.90 cm, y_e  -1.06 cm, th_e  1.84 deg, distance  1.40 cm, v 0.309, w 0.026
t =  3.35 s: x_e   0.63 cm, y_e  -0.78 cm, th_e  1.66 deg, distance  1.00 cm, v 0.306, w 0.332
t =  6.00 s: x_e   0.00 cm, y_e  -0.01 cm, th_e  0.07 deg, distance  0.01 cm, v 0.300, w 0.302
t = 11.90 s: x_e   0.00 cm, y_e  -0.01 cm, th_e  0.02 deg, distance  0.01 cm, v 0.300, w 0.000
largest wheel speed: 0.548 m/s; largest heading error: 6.99 deg
largest distance from 3.35 s on: 0.01 m; time of the largest heading error: 0.7000000000000001 s


Without its two feedforward parts (v_r cos θ_e and ω_r), the tracker must make all its speed from the error.

In [7]:
def no_feedforward(t, pose):
    xr, yr, thr, vr, wr = ref(t)
    xe, ye, the = error_frame(pose, (xr, yr, thr))
    return KX * xe, vr * (KY * ye + KTH * np.sin(the))
r0 = simulate(no_feedforward)
xe0_, ye0_, the0_, d0_ = errors(r0)
for t in (2, 5, 8, T_END):
    i = int(round(t / DT))
    print(f"t = {t:5.2f} s: x_e {xe0_[i]:.3f} m, y_e {ye0_[i]:.3f} m, th_e {np.degrees(the0_[i]):5.1f} deg")
print("largest heading error on the arc:", round(np.degrees(np.abs(the0_[int(T1 / DT):int(T2 / DT)]).max()), 1), "deg")

t =  2.00 s: x_e 0.287 m, y_e -0.030 m, th_e   4.2 deg
t =  5.00 s: x_e 0.289 m, y_e -0.045 m, th_e  11.4 deg
t =  8.00 s: x_e 0.273 m, y_e -0.067 m, th_e  14.9 deg
t = 11.90 s: x_e 0.297 m, y_e -0.019 m, th_e   2.7 deg
largest heading error on the arc: 15.1 deg


## 5. The Lyapunov function (Note, Section 5)

In [8]:
V = lyap(xe, ye, the)
rate = -KX * xe ** 2 - V_R * KTH * np.sin(the) ** 2 / KY
print("V at the start:", V[0], "m^2; its rate:", rate[0], "m^2/s")
print("V at 1 s:", round(V[100], 4), "; first time below 0.0001 m^2:", r["t"][np.argmax(V < 1e-4)], "s")
print("rate from the formula never above zero:", bool((rate <= 0).all()))
dV = np.diff(V)
print("largest one-step rise of V in the 20 Hz run:", dV.max(), "m^2, at t =", r["t"][dV.argmax()], "s (the arc starts at 3.333 s)")

V at the start: 0.025000000000000005 m^2; its rate: -0.04000000000000001 m^2/s
V at 1 s: 0.0046 ; first time below 0.0001 m^2: 3.15 s
rate from the formula never above zero: True
largest one-step rise of V in the 20 Hz run: 3.739168367281379e-06 m^2, at t = 3.34 s (the arc starts at 3.333 s)


The check at easy values: x_e = 0.2 m, y_e = −0.1 m, θ_e = 30°, on a straight (ω_r = 0).

In [9]:
xe_, ye_, th_ = 0.2, -0.1, np.radians(30)
v_ = V_R * np.cos(th_) + KX * xe_
w_ = 0 + V_R * (KY * ye_ + KTH * np.sin(th_))
xd = w_ * ye_ - v_ + V_R * np.cos(th_); yd = -w_ * xe_ + V_R * np.sin(th_); thd = 0 - w_
print("v, w:", round(v_, 4), round(w_, 4), "; rates:", round(xd, 4), round(yd, 4), round(thd, 4))
terms = (xe_ * xd, ye_ * yd, np.sin(th_) * thd / KY)
print("terms x_e x_e', y_e y_e', sin th_e th_e' / K_y:", np.round(terms, 5), " sum:", round(sum(terms), 5))
print("w x_e y_e =", round(w_ * xe_ * ye_, 4), "; v_r y_e sin th_e =", round(V_R * ye_ * np.sin(th_), 4))
print("formula -K_x x_e^2 - v_r K_th sin^2 / K_y =", round(-KX * xe_ ** 2 - V_R * KTH * np.sin(th_) ** 2 / KY, 5))

v, w: 0.4598 0.72 ; rates: -0.272 0.006 -0.72
terms x_e x_e', y_e y_e', sin th_e th_e' / K_y: [-0.0544 -0.0006 -0.0225]  sum: -0.0775
w x_e y_e = -0.0144 ; v_r y_e sin th_e = -0.015
formula -K_x x_e^2 - v_r K_th sin^2 / K_y = -0.0775


Damping on a straight, for small errors: y_e'' + v_r K_θ y_e' + v_r² K_y y_e = 0.

In [10]:
wn, two_zeta_wn = V_R * np.sqrt(KY), V_R * KTH
print("v_r^2 K_y =", V_R ** 2 * KY, "; v_r K_th =", two_zeta_wn, "; natural frequency", wn, "rad/s; damping ratio", two_zeta_wn / (2 * wn))
print("critically damped: 9.2 percent left at w_n t = 4:", round(5 * np.exp(-4), 4), "after", 4 / wn, "s, that is", V_R * 4 / wn, "m")
for kth in (4.0, 8.0, 16.0):
    rr = simulate(lambda t, p, kth=kth: kanayama(t, p, kth=kth))
    y = errors(rr)[1]
    bad = np.where(np.abs(y) >= 0.01)[0]
    print(f"K_th = {kth:4.1f} per m (damping {kth / (2 * np.sqrt(KY)):.1f}): largest crossing {100 * y.max():.2f} cm,"
          f" within 1 cm for good from {rr['t'][bad[-1] + 1]:.2f} s; wheel limit hit: {rr['clipped']}")

v_r^2 K_y = 1.44 ; v_r K_th = 2.4 ; natural frequency 1.2 rad/s; damping ratio 1.0
critically damped: 9.2 percent left at w_n t = 4: 0.0916 after 3.3333333333333335 s, that is 1.0 m
K_th =  4.0 per m (damping 0.5): largest crossing 1.07 cm, within 1 cm for good from 3.31 s; wheel limit hit: False
K_th =  8.0 per m (damping 1.0): largest crossing 0.06 cm, within 1 cm for good from 3.07 s; wheel limit hit: False
K_th = 16.0 per m (damping 2.0): largest crossing -0.10 cm, within 1 cm for good from 6.17 s; wheel limit hit: False


## 6. Feedback linearisation of an offset point (Note, Section 6)

In [11]:
v1, w1 = feedback_lin(0.0, START)
print("first command with P 0.2 m ahead: v =", round(v1, 3), "m/s, w =", round(w1, 3), "rad/s")
pr, prd = p_ref(0.0)
print("reference P at the start", pr, "moving at", prd, "; our P", p_point(START), "; wanted P velocity", prd + K_P * (pr - p_point(START)))
for d in (D_P, -D_P):
    rr = simulate(lambda t, p, d=d: feedback_lin(t, p, d=d))
    P = np.array([p_point((x, y, th), d) for x, y, th in zip(rr["x"], rr["y"], rr["th"])])
    PR = np.array([p_ref(t, d)[0] for t in rr["t"]])
    g = np.hypot(*(PR - P).T)
    mid = errors(rr)[3]
    print(f"d = {d:+.1f} m: P gap at 0, 1, 3 s: {g[0]:.4f}, {g[100]:.4f}, {g[300]:.4f} m; at the end {g[-1]:.5f} m;"
          f" final heading {np.degrees(rr['th'][-1]) % 360:.1f} deg, final v {rr['v'][-1]:+.3f} m/s;"
          f" axle midpoint to the timetable at the end {mid[-1]:.4f} m; wheel limit hit: {rr['clipped']}")
print("5 percent smaller every 0.05 s for 1 s:", round(0.2236 * 0.95 ** 20, 4), "; continuous e^-1:", round(0.2236 * np.exp(-1), 4))

first command with P 0.2 m ahead: v = 0.5 m/s, w = -0.5 rad/s
reference P at the start [0.2 0. ] moving at [0.3 0. ] ; our P [0.  0.1] ; wanted P velocity [ 0.5 -0.1]
d = +0.2 m: P gap at 0, 1, 3 s: 0.2236, 0.0800, 0.0104 m; at the end 0.00005 m; final heading 90.0 deg, final v +0.300 m/s; axle midpoint to the timetable at the end 0.0001 m; wheel limit hit: False


d = -0.2 m: P gap at 0, 1, 3 s: 0.2236, 0.0820, 0.0133 m; at the end 0.00013 m; final heading 269.8 deg, final v -0.300 m/s; axle midpoint to the timetable at the end 0.4000 m; wheel limit hit: False
5 percent smaller every 0.05 s for 1 s: 0.0802 ; continuous e^-1: 0.0823
